# Captuto PaddleOCR service
Aktifkan Internet dan GPU pada Kaggle sebelum menjalankan notebook ini. Tambahkan Kaggle Secrets `ZROK_ENABLE_TOKEN` dan `PADDLE_OCR_TOKEN`.

In [ ]:
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu paddleocr paddlex torch torchvision torchaudio
!pip install -q --no-cache-dir paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install -q --no-cache-dir paddleocr==3.7.0 paddlex==3.7.0 fastapi 'uvicorn[standard]' python-multipart
# ModelScope dipakai PaddleX untuk registri model; CPU-only PyTorch mencegah konflik NCCL Kaggle.
!pip install -q --force-reinstall --no-cache-dir torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q --force-reinstall --no-deps --no-cache-dir numpy==2.3.3
!python -c "import numpy; print('NumPy terpasang:', numpy.__version__)"
!python -m pip show paddlepaddle-gpu paddleocr paddlex
!curl -sSf https://get.openziti.io/install.bash | bash -s zrok2
!which zrok2

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
secrets = UserSecretsClient()
os.environ['ZROK_ENABLE_TOKEN'] = secrets.get_secret('ZROK_ENABLE_TOKEN')
os.environ['PADDLE_OCR_TOKEN'] = secrets.get_secret('PADDLE_OCR_TOKEN')

In [ ]:
# Tambahkan service.py dari folder paddleocr-kaggle sebagai Kaggle Input.
# Cell ini otomatis mencari file tersebut pada seluruh input yang sudah dipasang.
from pathlib import Path
candidates = [path for path in Path('/kaggle/input').glob('**/service.py') if path.is_file()]
if not candidates:
    raise FileNotFoundError('Tambahkan service.py sebagai Kaggle Input, lalu jalankan ulang cell ini.')
service_path = Path('/kaggle/working/service.py')
service_path.write_text(candidates[0].read_text())
print(f'Menggunakan {candidates[0]}')
!python -m py_compile /kaggle/working/service.py

In [ ]:
import json, os, socket, subprocess, sys, time, urllib.request
from pathlib import Path

def local_health():
    with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
        result = json.load(response)
    if result.get('status') != 'ok' or result.get('model') != 'PaddleOCR':
        raise RuntimeError('Port 8000 tidak menjalankan layanan Captuto PaddleOCR.')
    return result

def start_ocr_server():
    global server
    with socket.socket() as probe:
        probe.settimeout(2)
        occupied = probe.connect_ex(('127.0.0.1', 8000)) == 0
    if occupied:
        try:
            result = local_health()
        except Exception as exc:
            raise RuntimeError('Port 8000 terpakai tetapi belum sehat. Periksa proses lama/log; jangan menjalankan Uvicorn kedua.') from exc
        print('Menggunakan server yang sudah aktif:', result)
        return
    previous = globals().get('server')
    if previous is not None and previous.poll() is None:
        raise RuntimeError('Server sebelumnya masih startup. Tunggu dan periksa paddleocr.log sebelum menjalankan cell lagi.')
    if not os.environ.get('PADDLE_OCR_TOKEN'):
        raise RuntimeError('Jalankan cell Kaggle Secrets terlebih dahulu.')
    os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = '1'
    log_path = Path('/kaggle/working/paddleocr.log')
    with log_path.open('a') as log_file:
        server = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'service:app', '--host', '0.0.0.0', '--port', '8000'],
            cwd='/kaggle/working', stdout=log_file, stderr=subprocess.STDOUT, env=os.environ.copy(),
        )
    deadline = time.monotonic() + 300
    try:
        while time.monotonic() < deadline:
            if server.poll() is not None:
                raise RuntimeError('Uvicorn berhenti:\n' + log_path.read_text()[-4000:])
            try:
                result = local_health()
            except (OSError, ValueError):
                time.sleep(2)
                continue
            if result.get('pid') != server.pid:
                raise RuntimeError('Health berasal dari proses lain. Gunakan service.py terbaru dan periksa proses port 8000.')
            if server.poll() is not None:
                raise RuntimeError('Server berhenti setelah health check.')
            print('Server siap:', result)
            return
        raise TimeoutError('Model belum siap dalam 5 menit. Periksa paddleocr.log.')
    except BaseException:
        if server.poll() is None:
            server.terminate()
            try:
                server.wait(timeout=10)
            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()
        raise

start_ocr_server()

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path
zrok_bin = shutil.which('zrok2') or shutil.which('zrok')
if zrok_bin is None:
    raise FileNotFoundError('Executable zrok2/zrok tidak ditemukan di PATH. Periksa lokasi instalasi binary.')
print('Executable tunnel:', zrok_bin)
# Ubah menjadi True hanya untuk environment yang belum di-enable.
ENABLE_ENVIRONMENT = False
if ENABLE_ENVIRONMENT:
    token = os.environ.get('ZROK_ENABLE_TOKEN')
    if not token:
        raise RuntimeError('Jalankan cell Kaggle Secrets terlebih dahulu.')
    enabled = subprocess.run([zrok_bin, 'enable', '--headless', token], timeout=60)
    if enabled.returncode != 0:
        raise RuntimeError('Enable gagal. Periksa output zrok; jangan membuka tunnel sebelum environment siap.')
local_health()  # Jangan membuka tunnel ketika server lokal belum siap.
existing_tunnel = globals().get('tunnel')
if existing_tunnel is not None and existing_tunnel.poll() is None:
    print('Tunnel masih berjalan.')
else:
    with open('/kaggle/working/zrok.log', 'w') as tunnel_log:
        tunnel = subprocess.Popen(
            [zrok_bin, 'share', 'public', '--headless', 'http://127.0.0.1:8000'],
            stdout=tunnel_log, stderr=subprocess.STDOUT, text=True,
        )
log_path = Path('/kaggle/working/zrok.log')
print('Menunggu URL tunnel, maksimal 30 detik...')
for _ in range(30):
    log_text = log_path.read_text() if log_path.exists() else ''
    if tunnel.poll() is not None:
        raise RuntimeError('Tunnel berhenti:\n' + (log_text or '(log masih kosong)'))
    if 'https://' in log_text or 'access your zrok share at the following endpoints:' in log_text:
        break
    time.sleep(1)
print(log_text or 'Proses masih berjalan, tetapi belum menulis log. Periksa zrok.log lagi beberapa saat kemudian.')
# Salin URL persis dari log (share.zrok.io atau shares.zrok.io).
# Uji URL/health dari luar Kaggle sebelum upload dokumen.